# A2 – Semantic Chunking

- **Adapted from:** `all_rag_techniques/semantic_chunking.ipynb`
- **Experiment ID:** `A2_SEMANTIC_CHUNK`
- **Corpus:** `report_data/raw`
- **Evaluation set:** `report_data/evaluation/questions.json`
- **Purpose:** Compare semantic chunking with the best fixed chunk size from A1. Only the chunking strategy changes; retriever, Top-K, prompt, and LLM remain identical.

## Hypothesis

Semantic chunking should preserve logical units better than fixed-size, improving Recall@K and Context Precision on queries requiring complete reasoning chains. However, it may produce chunks with high variance in length.

## Control Variables

- Fixed chunk baseline: best from A1
- Retriever: dense only, same embedding model
- Top-K: same as A0/A1
- Prompt: same naive prompt
- LLM: same

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, save_csv_summary, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)

EXPERIMENT_ID = "A2_SEMANTIC_CHUNK"
NOTEBOOK = "03_semantic_chunking.ipynb"
SEED = config["seed"]
CONFIG_HASH = config["_config_hash"]
TOP_K = config["baseline"]["top_k"]

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)
print(f"LLM OK: {llm.invoke('hi').content[:30]}")
print(f"Embedding OK: dim={len(embeddings.embed_query('test'))}")

## 2. Load Corpus & Evaluation Set

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]
documents = []
for pdf_file in raw_data_path.glob("*.pdf"):
    documents.extend(PyPDFLoader(str(pdf_file)).load())
print(f"Loaded {len(documents)} pages.")

questions_path = PROJECT_ROOT / config["paths"]["questions"]
with open(questions_path, "r", encoding="utf-8") as f:
    eval_questions = json.load(f)
print(f"Loaded {len(eval_questions)} questions.")

## 3. Fixed Chunk Baseline (Best from A1)

Use the best chunk size found in A1 as the control group.

In [ ]:
# Set this to the best chunk_size from A1 results
BEST_FIXED_CHUNK_SIZE = 500  # UPDATE after running A1
BEST_FIXED_OVERLAP = BEST_FIXED_CHUNK_SIZE // 10

fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=BEST_FIXED_CHUNK_SIZE,
    chunk_overlap=BEST_FIXED_OVERLAP,
    length_function=len,
)
fixed_chunks = fixed_splitter.split_documents(documents)
for c in fixed_chunks:
    c.page_content = c.page_content.replace('\t', ' ')

print(f"Fixed chunks: {len(fixed_chunks)} (size={BEST_FIXED_CHUNK_SIZE}, overlap={BEST_FIXED_OVERLAP})")
print(f"  Avg length: {np.mean([len(c.page_content) for c in fixed_chunks]):.0f} chars")

## 4. Semantic Chunking

Split by detecting topic shifts via embedding cosine similarity between consecutive sentences.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=75,  # percentile threshold for breakpoints
)

# Combine all page text for semantic splitting
full_text = "\n\n".join([doc.page_content for doc in documents])
semantic_chunks_raw = semantic_splitter.create_documents([full_text])

# Clean
for c in semantic_chunks_raw:
    c.page_content = c.page_content.replace('\t', ' ')

semantic_chunks = semantic_chunks_raw

print(f"Semantic chunks: {len(semantic_chunks)}")
lengths = [len(c.page_content) for c in semantic_chunks]
print(f"  Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.0f}, Std: {np.std(lengths):.0f}")

## 5. Chunk Length Distribution

In [ ]:
print("\nChunk length distribution (semantic):")
bins = [0, 200, 500, 800, 1200, 2000, float('inf')]
labels = ["<200", "200-500", "500-800", "800-1200", "1200-2000", ">2000"]
for i in range(len(bins)-1):
    count = sum(1 for l in lengths if bins[i] <= l < bins[i+1])
    print(f"  {labels[i]:10s}: {count:3d} chunks ({count/len(lengths)*100:.1f}%)")

## 6. Run Evaluation (Both Strategies)

In [ ]:
NAIVE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""
)
chain = NAIVE_PROMPT | llm


def evaluate_chunking(chunks, strategy_name):
    """Run evaluation for a given set of chunks."""
    with Timer() as t_idx:
        vs = FAISS.from_documents(chunks, embeddings)
    retriever = vs.as_retriever(search_kwargs={"k": TOP_K})
    print(f"\n[{strategy_name}] Index built in {t_idx.elapsed:.2f}s ({len(chunks)} chunks)")

    results = []
    for q in eval_questions:
        with Timer() as t_ret:
            docs = retriever.invoke(q["question"])
        retrieved_ids = [d.metadata.get("source", f"chunk_{i}") for i, d in enumerate(docs)]
        context = "\n\n".join([d.page_content for d in docs])

        with Timer() as t_gen:
            response = chain.invoke({"context": context, "question": q["question"]})

        metrics = compute_retrieval_metrics(retrieved_ids, q.get("relevant_documents", []), k=TOP_K)
        record = build_result_record(
            experiment_id=f"{EXPERIMENT_ID}_{strategy_name}",
            notebook=NOTEBOOK,
            config_hash=CONFIG_HASH,
            seed=SEED,
            question_id=q["question_id"],
            question=q["question"],
            answer=response.content,
            latency={
                "retrieval_seconds": t_ret.elapsed,
                "generation_seconds": t_gen.elapsed,
                "total_seconds": t_ret.elapsed + t_gen.elapsed,
            },
            usage={"context_chars": len(context)},
            metrics=metrics,
            chunking_strategy=strategy_name,
            num_chunks=len(chunks),
        )
        results.append(record)
    return results

In [ ]:
fixed_results = evaluate_chunking(fixed_chunks, "FIXED")
semantic_results = evaluate_chunking(semantic_chunks, "SEMANTIC")

all_results = fixed_results + semantic_results

## 7. Comparison Table

In [ ]:
def summarize(results, name):
    metrics_agg = {}
    for key in results[0]["metrics"]:
        vals = [r["metrics"][key] for r in results if r["metrics"].get(key) is not None]
        metrics_agg[key] = round(np.mean(vals), 3) if vals else None
    return {
        "strategy": name,
        "num_chunks": results[0].get("num_chunks", "?"),
        "avg_context_chars": int(np.mean([r["usage"]["context_chars"] for r in results])),
        "avg_latency_s": round(np.mean([r["latency"]["total_seconds"] for r in results]), 3),
        **metrics_agg,
    }

comparison = [
    summarize(fixed_results, f"Fixed ({BEST_FIXED_CHUNK_SIZE})"),
    summarize(semantic_results, "Semantic"),
]

df = pd.DataFrame(comparison)
print(df.to_string(index=False))

## 8. Save Results

In [ ]:
output_dir = PROJECT_ROOT / config["paths"]["results"]
save_jsonl(all_results, output_dir / "A2_semantic_chunking.jsonl")
save_csv_summary(comparison, output_dir / "A2_semantic_chunking_summary.csv")
save_config_snapshot(config, output_dir)

## 9. Observations

- Semantic chunking produced `___` chunks vs `___` fixed chunks.
- Chunk length variance: semantic has std=`___` vs fixed std≈0.
- Recall@K: semantic `___` vs fixed `___`.
- Very short semantic chunks (<100 chars) may indicate over-splitting.
- Very long semantic chunks (>2000 chars) may indicate under-splitting.
- The best chunking strategy from A1/A2 will be used for A3+ experiments.

_Fill in after running._